# Mission Control Center (MCC) per ground station: compiti, architettura, tecnologie
---

Un **Mission Control Center (MCC)** è l’insieme di **procedure e sistemi software** che, da terra, monitorano e comandano un veicolo spaziale o un lanciatore durante una missione. E' un **sistema distribuito** che acquisisce la telemetria, la decodifica, la rende disponibile in tempo reale, e consente di inviare telecomandi verso il veicolo.

Nel contesto di una ground station per razzi sonda o CubeSat, l’MCC è il livello che sta **sopra** l’hardware radio (antenna, SDR, amplificatori) e **sotto** l’interfaccia umana (console, dashboard). È il punto in cui il segnale radio diventa dato comprensibile, e il comando umano diventa pacchetto trasmissibile.

> **Osservazione 0.2 — MCC vs ground station.**  
> I due termini vengono spesso usati in modo intercambiabile, ma hanno ruoli distinti. La **ground station** è l’infrastruttura fisica di comunicazione: antenna, radio, computer di controllo, link di rete. L’**MCC** è il livello logico di **supervisione e comando**: elabora la telemetria, gestisce gli allarmi, registra i dati, presenta lo stato, e orchestra le operazioni. 

---

## 1. Compiti di un MCC

### Definizione 1.1 — Le funzioni minime

Un MCC, indipendentemente dalla scala, deve svolgere sei funzioni fondamentali. Le prime tre sono **essenziali** (senza di esse non c’è missione); le altre tre sono **operative** (senza di esse la missione è possibile ma fragile).

1. **Acquisizione e decodifica della telemetria (TM)**: ricevere i pacchetti dal segmento di terra, verificarne l’integrità (CRC), riassemblare i frame, e convertirli da unità grezze (count, raw ADC) a grandezze fisiche (Volt, °C, bar, m/s)[reference:2].

2. **Visualizzazione in tempo reale**: presentare i parametri su console, grafici a strip-chart, indicatori di stato, mappe di missione. La latenza di visualizzazione è un requisito critico: un operatore che vede un parametro in ritardo di 2 secondi sta prendendo decisioni su un passato che non esiste più.

3. **Telecomando (TC)**: costruire, validare e inviare comandi al veicolo, con conferma di esecuzione (command acknowledgement) e gestione delle sequenze (command sequencing).

4. **Allarmi e gestione delle anomalie**: confrontare i parametri con soglie (limiti di sicurezza), generare allarmi, e supportare la procedura di recovery. Un MCC senza gestione allarmi è un visualizzatore, non un centro di controllo.

5. **Registrazione e replay**: loggare ogni pacchetto, ogni comando, ogni azione dell’operatore, con timestamp. Il replay è essenziale per l’analisi post-missione.

6. **Gestione delle operazioni**: (procedure runner, scheduler, database di missione).

> **Osservazione 1.2 — Il principio di separazione dei ruoli.**  
> In un MCC maturo, le sei funzioni sono **servizi separati** che comunicano tramite interfacce definite. Questo permette di sostituire un componente senza riscrivere gli altri, di scalare orizzontalmente (più consumer di telemetria), e di testare ogni servizio in isolamento.

### Osservazione 1.3 — Compiti specifici per razzi sonda vs CubeSat

La natura del veicolo cambia il **profilo temporale** dell’MCC, e quindi le priorità tecnologiche.

| Aspetto | Razzo sonda (sounding rocket) | CubeSat |
|---|---|---|
| **Durata della missione** | minuti (5–20 min) | mesi–anni |
| **Finestra di comunicazione** | continua durante il volo | passaggi orbitali di 5–15 min, alcune volte al giorno |
| **Frequenza di telemetria** | alta (decine–centinaia di Hz) | bassa (beacon ogni minuto, downlink a pacchetti) |
| **Latenza richiesta** | millisecondi (controllo in volo) | secondi–minuti (comandi programmati) |
| **Priorità** | visualizzazione real-time, safety, abort | pianificazione passaggi, store-and-forward, analisi trend |
| **Tecnologia tipica** | bus real-time, stream processing, GUI a bassa latenza | database di missione, scheduler, dashboard web |

Per un razzo sonda, l’MCC è un **sistema real-time** che deve reggere raffiche di telemetria e produrre decisioni in pochi millisecondi. Per un CubeSat, l’MCC è un **sistema di gestione di dati asincroni**: riceve pacchetti durante i passaggi, li archivia, li rende disponibili per l’analisi, e pianifica i comandi per il passaggio successivo. La tecnologia cambia di conseguenza.

---

## 2. Architettura di riferimento

### Definizione 2.1 — I quattro strati

Un’architettura MCC moderna e modulare si organizza in quattro strati, dal più vicino all’hardware al più vicino all’uomo.

4. Presentation │ Dashboard, console, allarmi, report

3. Mission logic │ Comandi, procedure, pianificazione, FDIR

2. Data handling │ Decodifica, calibrazione, archiviazione

1. Link & transport │ Radio, SDR, framing, protocolli (CCSDS)


**Strato 1 — Link & transport.** È il livello della ground station fisica: antenna, SDR (Software Defined Radio), demodulazione, deframing. Il suo output è un flusso di **frame** o **pacchetti** grezzi. Nel caso CubeSat, uno degli hardware più usati è il **Raspberry Pi 5** come computer di controllo della ground station, affiancato a un SDR (es. HackRF One) e antenne Yagi-Uda per le frequenze amateur (146 MHz, 437 MHz).

**Strato 2 — Data handling.** È il cuore “informatico” dell’MCC: prende i pacchetti grezzi, verifica il CRC, riassembla i frame, decodifica i parametri secondo un **database di missione** (MDB), applica le calibrazioni, e produce valori in grandezze fisiche. Questo strato è tipicamente un **servizio** (daemon) che pubblica i valori decodificati su un bus interno.

**Strato 3 — Mission logic.** Contiene la logica operativa: validazione dei comandi, sequenze, procedure, e nei sistemi più maturi **FDIR** (Fault Detection, Isolation, Recovery). È lo strato che trasforma “dati” in “decisioni”.

**Strato 4 — Presentation.** Dashboard, console, allarmi, report. È l’unico strato che l’operatore vede direttamente. La sua qualità determina l’usabilità dell’intero sistema.

> **Osservazione 2.2 — Il bus interno è la scelta architetturale più importante.**  
> Il modo in cui gli strati comunicano tra loro determina la flessibilità del sistema. Le tre opzioni principali sono: 
>
> (a) **chiamate dirette** tra moduli (semplice ma accoppiato)
> (b) **broker pub/sub** (disaccoppiato, adatto a telemetria multi-consumer)
> (c) **code persistenti** (Kafka, Redis Streams; robusto ma complesso). Per un MCC di piccola-media scala, il **pub/sub con MQTT** è il compromesso più usato: leggero, standard, con QoS configurabile, e con client in ogni linguaggio.

---

## 3. Tecnologie per l’implementazione

### Definizione 3.3 — Componenti software tipici

| Componente | Ruolo | Tecnologie tipiche |
|---|---|---|
| **Decoder di telemetria** | Da frame grezzi a parametri ingegneristici | Python (con `construct`, `bitstring`), C/C++, Rust |
| **Broker di messaggi** | Bus pub/sub interno | **MQTT** (Mosquitto, EMQX), Redis Pub/Sub |
| **Mission control framework** | Gestione TM/TC, archivio, allarmi, procedure | **Yamcs**, **COSMOS (OpenC3)**, **Open MCT** |
| **Time-series database** | Archivio storico, query temporali | **InfluxDB**, TimescaleDB |
| **Visualizzazione** | Dashboard, console, grafici | **Grafana**, Open MCT, React |
| **Service management** | Avvio, supervisione, restart dei servizi | **systemd** |
| **Containerizzazione** | Isolamento, riproducibilità, deploy | **Docker**, Podman |

**Yamcs** è un framework open-source per mission control, scritto in Java, con supporto nativo per CCSDS, CFDP, XTCE, e un’architettura modulare a servizi (telemetria, comandi, archivio, allarmi). È stato usato come base per l’MCC open-source del DLR, esteso con plugin per verifica CRC, riassemblaggio pacchetti e command sequencing, e validato su un volo reale con latenze end-to-end di ~550 ms.

**COSMOS (OpenC3)** è un framework open-source, flight-proven, per comando e controllo di hardware reale, con supporto per seriale, TCP, UDP, HTTP, MQTT e CCSDS, e con un’interfaccia a dashboard e script runner.

**Open MCT** (NASA Ames) è una piattaforma di visualizzazione per telemetria real-time, con il concetto di **Time Conductor** che sincronizza tutti i dati a un riferimento temporale comune, requisito critico per il monitoraggio di veicoli spaziali.

**InfluxDB + Grafana** è la coppia più diffusa per la visualizzazione di telemetria: InfluxDB come time-series database, Grafana come dashboard. È usata in progetti come SatNOGS per il monitoraggio delle telemetrie decodificate.

> **Osservazione 3.4 — Il pattern architetturale emergente.**  
> I progetti reali convergono su un’architettura **a servizi containerizzati** con un **broker pub/sub** come bus interno. Il DLR descrive esplicitamente “containerized services, broker bridging for publish–subscribe communication” come scelta di fondo del suo MCC open-source. Questo pattern permette di: 
>
>(a) sviluppare e testare ogni servizio in isolamento 
>(b) sostituire un componente senza impattare gli altri
>(c) deployare su hardware diverso (Raspberry Pi, server, cloud) senza cambiare il codice.

---

## 4. Requisiti non funzionali

### Osservazione 4.1 — I requisiti che contano più delle feature

Un MCC che “fa tutto” ma perde pacchetti o va in crash durante un passaggio critico è peggio di un MCC che fa meno ma è affidabile. I requisiti non funzionali sono, in ordine di priorità:

1. **Affidabilità**: nessuna perdita di telemetria critica. Meccanismi: QoS 1/2 su MQTT, sessioni persistenti, archiviazione su disco prima dell’elaborazione, ridondanza.

2. **Latenza**: il tempo tra ricezione del pacchetto e visualizzazione deve essere noto e stabile. Un jitter elevato è peggio di una latenza media alta ma costante.

3. **Osservabilità**: ogni servizio deve loggare in modo strutturato (timestamp, livello, contesto), e deve essere monitorabile (metriche, health check). Senza osservabilità, il debug durante una missione è impossibile.

4. **Riproducibilità**: l’ambiente deve essere ricostruibile da zero (Docker, Ansible).

5. **Sicurezza**: autenticazione, autorizzazione, cifratura del link. In un contesto di lancio, un comando non autorizzato è un rischio.

---

## 5. MCC “leggero”

### Osservazione 5.1 — Esistono MCC di dimensioni diverse

Non tutti gli MCC sono uguali. Una tassonomia utile:


| Scala | Esempio | Hardware | Software |
|---|---|---|---|---|
| **Full-scale** | ESA/NASA/DLR MCC | Server cluster, sale controllo | Yamcs/COSMOS + database + VPN | 
| **Small mission** | MCC CubeSat | 1–2 server o Raspberry Pi 5 | MQTT + InfluxDB + Grafana + decoder custom | 
| **Portable** | Ground station da campo per razzo | Raspberry Pi 5 + SDR | Servizi Python + systemd |


> **Osservazione 5.2 — Il principio guida per un MCC leggero.**  
> La scelta architetturale fondamentale è: **usare componenti standard e disaccoppiati, e concentrare la complessità dove serve**. Un broker MQTT, un time-series database, una dashboard Grafana, e servizi Python specializzati coprono l’80% delle esigenze di un MCC di piccola scala. Il restante 20% (protocolli proprietari, procedure di missione specifiche, integrazione con hardware non standard) è dove si scrive codice.
